[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/iap/blob/main/04_Redes_Profundas.ipynb)

# UNIVERSIDADE FEDERAL FLUMINENSE

**Introdução ao Aprendizado Profundo**

**Prof.: Diogo Ferreira de Lima Silva**

---
# AULA 04 — CLASSIFICAÇÃO E REDES PROFUNDAS
---

Até agora a rede previa um **número** (regressão). Muitos problemas práticos, porém, pedem uma **categoria**: a peça é defeituosa ou não? O cliente vai cancelar ou não? Que dígito está escrito nesta imagem?

**O que você vai aprender**

- usar uma rede para **classificar** (duas classes e várias classes);
- medir o desempenho com a **acurácia**;
- construir redes **profundas** (com várias camadas ocultas);
- treinar com **lotes** de dados usando o `DataLoader`;
- reconhecer dígitos manuscritos no conjunto **MNIST**.

## 0. Importando as bibliotecas

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)      # fixa a semente: os resultados ficam iguais a cada execução

# cores usadas nos gráficos
LARANJA, AZUL, VERDE, VERMELHO, CINZA = "#E08D2B", "#1F7A8C", "#2E7D4F", "#B3151A", "#5B6472"

---
# PARTE 1 — Classificação com duas classes
---

## 1. Dados

O scikit-learn tem um gerador de dados clássico, `make_moons`: pontos no plano divididos em duas "luas" entrelaçadas. Cada ponto tem **duas** características ($x_1$ e $x_2$) e pertence à classe **0** ou à classe **1**.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=600, noise=0.25, random_state=42)

print("formato de X:", X.shape)       # 600 pontos, 2 características
print("primeiros rótulos:", y[:10])   # 0 ou 1

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(X_tr[y_tr == 0, 0], X_tr[y_tr == 0, 1], color=AZUL, s=15, label="classe 0")
plt.scatter(X_tr[y_tr == 1, 0], X_tr[y_tr == 1, 1], color=LARANJA, s=15, label="classe 1")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.legend(); plt.grid(alpha=0.3)
plt.show()


## 2. Tensores

- `X` já tem duas colunas, então **não** precisa de `reshape`;
- `y` vira uma coluna `(N, 1)` com valores `0.0` ou `1.0`.

In [ ]:
X_treino = torch.tensor(X_tr, dtype=torch.float32)
Y_treino = torch.tensor(y_tr, dtype=torch.float32).reshape(-1, 1)
X_teste  = torch.tensor(X_te, dtype=torch.float32)
Y_teste  = torch.tensor(y_te, dtype=torch.float32).reshape(-1, 1)

print("X_treino:", X_treino.shape, "| Y_treino:", Y_treino.shape)

## 3. Modelo: uma rede profunda

Uma rede **profunda** é simplesmente uma rede com **mais de uma camada oculta**. Em código, basta empilhar mais pares `nn.Linear` + `nn.ReLU`:

- a entrada tem **2** números ($x_1$, $x_2$), então a primeira camada é `nn.Linear(2, ...)`;
- a saída é **1** número, que indica a classe.

In [ ]:
rede = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),     # 1ª camada oculta: 16 neurônios
    nn.Linear(16, 16), nn.ReLU(),    # 2ª camada oculta: 16 neurônios
    nn.Linear(16, 1),                # saída: 1 número
)
print(rede)

## 4. Perda e otimizador

Como transformar a saída da rede (um número qualquer, positivo ou negativo) em uma classe?

1. A função **sigmoide** "espreme" qualquer número para o intervalo entre 0 e 1. Interpretamos o resultado como a **probabilidade** de o ponto ser da classe 1.
2. Se a probabilidade passar de 0,5, dizemos que é classe 1; senão, classe 0.

In [ ]:
z = torch.linspace(-6, 6, 100)

plt.figure(figsize=(5, 3))
plt.plot(z, torch.sigmoid(z), color=AZUL, linewidth=3)
plt.axhline(0.5, color=CINZA, linestyle=":")
plt.title("sigmoide: transforma qualquer número em probabilidade"); plt.grid(alpha=0.3)
plt.show()

Para treinar, usamos a perda **`nn.BCEWithLogitsLoss`** (entropia cruzada binária). Ela aplica a sigmoide por dentro e mede o quão longe a probabilidade prevista está do rótulo correto. Por isso **não** colocamos a sigmoide dentro da rede.

| Tarefa | Saída da rede | Perda |
|---|---|---|
| regressão (prever um número) | 1 número | `nn.MSELoss()` |
| classificação com 2 classes | 1 número | `nn.BCEWithLogitsLoss()` |

In [ ]:
perda_fn = nn.BCEWithLogitsLoss()
otimizador = torch.optim.Adam(rede.parameters(), lr=0.01)

## 5. Laço de treino

Exatamente o mesmo de sempre.

In [ ]:
for epoca in range(1000):
    saida = rede(X_treino)
    perda = perda_fn(saida, Y_treino)
    otimizador.zero_grad()
    perda.backward()
    otimizador.step()

    if epoca % 200 == 0:
        print(f"época {epoca:4d} | perda = {perda.item():.4f}")

## 6. Avaliação: a acurácia

Em classificação, a medida mais intuitiva é a **acurácia**: a fração de exemplos classificados corretamente.

In [ ]:
with torch.no_grad():
    probabilidade = torch.sigmoid(rede(X_teste))     # número entre 0 e 1
    classe_prevista = (probabilidade > 0.5).float()   # 1.0 se passar de 0,5; senão 0.0

acertos = (classe_prevista == Y_teste).float()      # 1 onde acertou, 0 onde errou
print("acurácia no teste:", acertos.mean().item())

### Vendo o que a rede aprendeu

A função abaixo pinta cada região do plano com a classe que a rede prevê ali. **Não é preciso entender o código desta função**: ela só desenha o gráfico.

In [ ]:
def desenha_fronteira(modelo, X, y, titulo=""):
    # cria uma grade de pontos cobrindo o plano e pede a previsão da rede em cada um
    g1, g2 = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    grade = torch.tensor(np.c_[g1.ravel(), g2.ravel()], dtype=torch.float32)
    with torch.no_grad():
        prob = torch.sigmoid(modelo(grade)).reshape(g1.shape)
    plt.contourf(g1, g2, prob, levels=[0, 0.5, 1], colors=[AZUL, LARANJA], alpha=0.25)
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color=AZUL, s=10)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color=LARANJA, s=10)
    plt.title(titulo)


plt.figure(figsize=(6, 4.5))
desenha_fronteira(rede, X_te, y_te, "Fronteira de decisão (dados de teste)")
plt.show()

## 7. Rasa ou profunda?

Vamos comparar três redes: **sem** camada oculta, com **uma** e com **três**. Para isso, uma pequena função que monta a rede com o número de camadas ocultas que pedirmos:

In [ ]:
def criar_rede(n_camadas_ocultas, neuronios=16):
    camadas = []
    entrada = 2
    for i in range(n_camadas_ocultas):
        camadas.append(nn.Linear(entrada, neuronios))
        camadas.append(nn.ReLU())
        entrada = neuronios
    camadas.append(nn.Linear(entrada, 1))      # camada de saída
    return nn.Sequential(*camadas)             # o * "desempacota" a lista



print(criar_rede(0))
print('-------------------------------')
print(criar_rede(1))
print('-------------------------------')
print(criar_rede(3))

In [ ]:
def treinar_classificador(modelo, epocas=1000):
    perda_fn = nn.BCEWithLogitsLoss()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    for epoca in range(epocas):
        perda = perda_fn(modelo(X_treino), Y_treino)
        otimizador.zero_grad()
        perda.backward()
        otimizador.step()


def acuracia(modelo, X, Y):
    with torch.no_grad():
        classe = (torch.sigmoid(modelo(X)) > 0.5).float()
    return (classe == Y).float().mean().item()


plt.figure(figsize=(16, 4))
for i, n in enumerate([0, 1, 3]):
    torch.manual_seed(42)
    modelo = criar_rede(n)
    treinar_classificador(modelo)
    plt.subplot(1, 3, i + 1)
    desenha_fronteira(modelo, X_te, y_te, f"{n} camada(s) oculta(s) | acurácia = {acuracia(modelo, X_teste, Y_teste):.1%}")
plt.show()

- **Sem camada oculta**, a rede só consegue traçar uma **linha reta**: é o limite de um modelo linear.
- Com camadas ocultas, a fronteira se **dobra** e acompanha o formato das luas.
- Mais camadas **nem sempre** significam mais acurácia. Modelos muito complexos para problemas simples podem gerar overfitting.

Agora vamos a um problema de verdade.

---
# PARTE 2 — Reconhecendo dígitos: o MNIST
---

O **MNIST** é o conjunto de dados mais famoso do aprendizado profundo: 70 mil imagens de dígitos escritos à mão (0 a 9), cada uma com $28 \times 28$ pixels.

## 1. Dados

A biblioteca `torchvision` (irmã do PyTorch) baixa o MNIST automaticamente. O argumento `transform=transforms.ToTensor()` converte cada imagem em um tensor com valores entre 0 e 1.

In [ ]:
from torchvision import datasets, transforms

mnist_treino = datasets.MNIST(root="dados", train=True,  download=True, transform=transforms.ToTensor())
mnist_teste  = datasets.MNIST(root="dados", train=False, download=True, transform=transforms.ToTensor())

print("imagens de treino:", len(mnist_treino))
print("imagens de teste :", len(mnist_teste))

Cada elemento do conjunto é um par **(imagem, rótulo)**:

In [ ]:
imagem, rotulo = mnist_treino[0]
print("formato da imagem:", imagem.shape)    # 1 canal (tons de cinza), 28 x 28 pixels
print("rótulo:", rotulo)

In [ ]:
plt.figure(figsize=(12, 2.5))
for i in range(10):
    imagem, rotulo = mnist_treino[i]
    plt.subplot(1, 10, i + 1)
    plt.imshow(imagem[0], cmap="gray_r")
    plt.title(rotulo); plt.axis("off")
plt.show()

## 2. Lotes de dados: o `DataLoader`

São 60 mil imagens. Em vez de passar todas pela rede de uma vez, dividimos os dados em **lotes** (*batches*) pequenos, por exemplo de 64 imagens. A cada lote, a rede dá um passo de correção. Isso é mais rápido e usa menos memória.

O `DataLoader` faz essa divisão para nós (e embaralha os dados a cada época com `shuffle=True`):

In [ ]:
from torch.utils.data import DataLoader

carregador_treino = DataLoader(mnist_treino, batch_size=64, shuffle=True)
carregador_teste  = DataLoader(mnist_teste,  batch_size=1000)

imagens, rotulos = next(iter(carregador_treino))     # pega o primeiro lote, só para ver
print("um lote de imagens:", imagens.shape)          # 64 imagens de 1 x 28 x 28
print("um lote de rótulos:", rotulos.shape)
print("número de lotes por época:", len(carregador_treino))

## 3. Modelo

A rede recebe números em fila, não imagens. Por isso, a primeira peça é `nn.Flatten()`, que "achata" cada imagem $28 \times 28$ em uma fila de $784$ números.

A saída tem **10** números, um para cada dígito. O dígito com o **maior** número é a previsão da rede.

In [ ]:
torch.manual_seed(42)
rede_mnist = nn.Sequential(
    nn.Flatten(),                      # 1 x 28 x 28  ->  784
    nn.Linear(784, 128), nn.ReLU(),    # 1ª camada oculta
    nn.Linear(128, 64), nn.ReLU(),     # 2ª camada oculta
    nn.Linear(64, 10),                 # 10 saídas: uma para cada dígito
)
print(rede_mnist)

## 4. Perda e otimizador

Para **mais de duas classes**, a perda é a **`nn.CrossEntropyLoss`** (entropia cruzada). Ela recebe os 10 números da rede e o rótulo correto (um inteiro de 0 a 9).

| Tarefa | Saída da rede | Perda |
|---|---|---|
| regressão | 1 número | `nn.MSELoss()` |
| 2 classes | 1 número | `nn.BCEWithLogitsLoss()` |
| várias classes | 1 número por classe | `nn.CrossEntropyLoss()` |

In [ ]:
perda_fn = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(rede_mnist.parameters(), lr=0.001)

## 5. Laço de treino com lotes

A única mudança no laço: agora há um **segundo `for`**, que percorre os lotes. Uma **época** é uma passada completa por todos os lotes.

> Esta célula leva cerca de 10 a 20 segundos.

In [ ]:
for epoca in range(3):
    for imagens, rotulos in carregador_treino:      # percorre os lotes
        saida = rede_mnist(imagens)
        perda = perda_fn(saida, rotulos)
        otimizador.zero_grad()
        perda.backward()
        otimizador.step()

    print(f"época {epoca + 1} concluída | perda no último lote = {perda.item():.4f}")

## 6. Avaliação

Percorremos os lotes do teste contando os acertos. O método `.argmax(dim=1)` devolve a posição do maior dos 10 números, ou seja, o dígito previsto.

In [ ]:
acertos = 0
total = 0
with torch.no_grad():
    for imagens, rotulos in carregador_teste:
        saida = rede_mnist(imagens)
        previsto = saida.argmax(dim=1)
        acertos = acertos + (previsto == rotulos).sum().item()
        total = total + len(rotulos)

print(f"acurácia no teste: {acertos / total:.2%}  ({acertos} de {total} imagens)")

Com poucas linhas de código e menos de um minuto de treino, a rede acerta cerca de **97%** das imagens de teste.

### Vendo algumas previsões

Título em verde: acerto. Em vermelho: erro.

In [ ]:
imagens, rotulos = next(iter(carregador_teste))
with torch.no_grad():
    previstos = rede_mnist(imagens).argmax(dim=1)

plt.figure(figsize=(12, 5))
for i in range(24):
    plt.subplot(3, 8, i + 1)
    plt.imshow(imagens[i][0], cmap="gray_r")
    cor = VERDE if previstos[i] == rotulos[i] else VERMELHO
    plt.title(f"prev: {previstos[i].item()}", color=cor); plt.axis("off")
plt.tight_layout(); plt.show()

E alguns **erros**, para entendermos onde a rede tem dificuldade:

In [ ]:
erros = (previstos != rotulos).nonzero().flatten()     # posições onde a rede errou

plt.figure(figsize=(12, 2.5))
for j, i in enumerate(erros[:8]):
    plt.subplot(1, 8, j + 1)
    plt.imshow(imagens[i][0], cmap="gray_r")
    plt.title(f"real {rotulos[i].item()} | prev {previstos[i].item()}", color=VERMELHO, fontsize=9)
    plt.axis("off")
plt.show()

Muitos erros são dígitos mal escritos, difíceis até para nós.

---
## Exercício
---

**1.** Na Parte 1, troque `noise=0.25` por `noise=0.5` em `make_moons`. A acurácia cai? Por quê?

**2.** Na Parte 2, treine a rede por 5 épocas em vez de 3. A acurácia no teste melhora?

**3.** Ainda na Parte 2, remova a segunda camada oculta (deixe `nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)`). A acurácia muda muito?

**4.** Troque `datasets.MNIST` por `datasets.FashionMNIST` (imagens de roupas e calçados, mesmo formato). A rede acerta mais ou menos?